# Comparing Checkov scanning modes: static directory scan vs Terraform plan JSON scan

Checkov can scan infrastructure-as-code in two main ways: pointing it at a directory of `.tf` files (static scan), or feeding it a Terraform plan JSON output (plan scan). This notebook walks through both, compares the results, and notes where each mode shines.

## Purpose

Understand the practical difference between Checkov's two scanning modes:
- **Static directory scan** — Checkov parses raw `.tf` files directly
- **Plan JSON scan** — You run `terraform plan -out=tfplan` + `terraform show -json tfplan`, then feed the JSON to Checkov

Plan scanning catches runtime-evaluated values (like `var.environment` in resource names) that static scanning misses, but it adds a dependency on `terraform` being available.

## Prerequisites

- Python 3.8+
- `checkov` installed (`pip install checkov`)
- `terraform` CLI available for plan generation
- A sample Terraform project with intentional misconfigurations

In [ ]:
# Check that checkov is available
import subprocess
import json
import os
import tempfile
from pathlib import Path

def check_prerequisites():
    """Verify checkov and terraform are accessible."""
    for cmd in ["checkov", "terraform"]:
        result = subprocess.run(["which", cmd], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"WARNING: {cmd} not found in PATH — some cells will fail")
        else:
            print(f"OK: {cmd} found at {result.stdout.strip()}")

check_prerequisites()

## Step 1: Create a sample Terraform project

I'll create a small Terraform project with a few intentional issues: an open security group, a public S3 bucket, and an unencrypted RDS instance.

In [ ]:
SAMPLE_TF = '''
provider "aws" {
  region = var.region
}

variable "region" {
  default = "us-east-1"
}

variable "environment" {
  default = "dev"
}

resource "aws_s3_bucket" "data" {
  bucket = "${var.environment}-data-bucket"
  acl    = "public-read"
}

resource "aws_security_group" "web" {
  name_prefix = "${var.environment}-"

  ingress {
    from_port   = 22
    to_port     = 22
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

resource "aws_db_instance" "main" {
  engine         = "postgres"
  instance_class = "db.t3.micro"
  storage_encrypted = false
}
'''

# Write the Terraform files to a temp directory
tf_dir = Path(tempfile.mkdtemp(prefix="checkov_compare_"))
main_tf = tf_dir / "main.tf"
main_tf.write_text(SAMPLE_TF)
print(f"Terraform project created at: {tf_dir}")
print(f"Files: {list(tf_dir.iterdir())}")

## Step 2: Static directory scan

This is the simplest approach — point Checkov at the directory and let it parse raw `.tf` files.

In [ ]:
def run_static_scan(directory: Path) -> dict:
    """Run Checkov static scan on a directory of .tf files.
    
    Returns parsed JSON output.
    """
    result = subprocess.run(
        ["checkov", "--directory", str(directory),
         "--framework", "terraform",
         "--output", "json",
         "--compact"],
        capture_output=True, text=True
    )
    if result.returncode not in (0, 1):
        print(f"ERROR: Checkov exited with code {result.returncode}")
        print(result.stderr[:500])
        return {"summary": {}, "results": []}
    
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print("Raw output (first 300 chars):", result.stdout[:300])
        return {"summary": {}, "results": []}


static_results = run_static_scan(tf_dir)
print(f"Static scan complete. Found {len(static_results.get('results', []))} results")

In [ ]:
# Extract failed checks from static scan
static_failed = []
for resource_type in static_results.get("results", []):
    if isinstance(resource_type, dict) and "passed_checks" in resource_type:
        static_failed.extend(resource_type.get("failed_checks", []))
    elif isinstance(resource_type, list):
        for item in resource_type:
            if isinstance(item, dict) and "check_id" in item:
                static_failed.append(item)

print(f"Failed checks (static scan): {len(static_failed)}")
for fc in static_failed[:10]:
    print(f"  - {fc.get('check_id', '?'):25s} {fc.get('check_name', '?')[:60]}")

## Step 3: Terraform plan JSON scan

For plan scanning, I need to run `terraform init`, `terraform plan`, and convert the plan to JSON.

In [ ]:
def generate_plan_json(tf_dir: Path) -> Path | None:
    """Initialize Terraform, generate a plan, and convert to JSON.
    
    Returns path to the plan JSON file, or None on failure.
    """
    # terraform init
    init = subprocess.run(["terraform", "init"], cwd=tf_dir,
                          capture_output=True, text=True)
    if init.returncode != 0:
        print(f"terraform init failed:\n{init.stderr[:500]}")
        return None
    
    # terraform plan -out=tfplan
    plan_file = tf_dir / "tfplan"
    plan = subprocess.run(["terraform", "plan", "-out=tfplan"],
                         cwd=tf_dir, capture_output=True, text=True)
    if plan.returncode != 0:
        print(f"terraform plan failed:\n{plan.stderr[:500]}")
        return None
    
    # terraform show -json tfplan > plan.json
    plan_json = tf_dir / "plan.json"
    show = subprocess.run(
        ["terraform", "show", "-json", str(plan_file)],
        cwd=tf_dir, capture_output=True, text=True
    )
    if show.returncode != 0:
        print(f"terraform show failed:\n{show.stderr[:500]}")
        return None
    
    plan_json.write_text(show.stdout)
    print(f"Plan JSON written to {plan_json}")
    return plan_json


plan_json_path = generate_plan_json(tf_dir)

In [ ]:
def run_plan_scan(plan_json_path: Path) -> dict:
    """Run Checkov plan scan on a Terraform plan JSON file."""
    result = subprocess.run(
        ["checkov", "--plan", str(plan_json_path),
         "--output", "json", "--compact"],
        capture_output=True, text=True
    )
    if result.returncode not in (0, 1):
        print(f"Checkov plan scan error: exit {result.returncode}")
        print(result.stderr[:500])
        return {"summary": {}, "results": []}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"summary": {}, "results": []}


if plan_json_path:
    plan_results = run_plan_scan(plan_json_path)
    print(f"Plan scan complete")
else:
    plan_results = {"summary": {}, "results": []}
    print("Skipping plan scan — plan JSON not available")

In [ ]:
# Extract failed checks from plan scan
plan_failed = []
for resource_type in plan_results.get("results", []):
    if isinstance(resource_type, dict) and "passed_checks" in resource_type:
        plan_failed.extend(resource_type.get("failed_checks", []))
    elif isinstance(resource_type, list):
        for item in resource_type:
            if isinstance(item, dict) and "check_id" in item:
                plan_failed.append(item)

print(f"Failed checks (plan scan): {len(plan_failed)}")
for fc in plan_failed[:10]:
    print(f"  - {fc.get('check_id', '?'):25s} {fc.get('check_name', '?')[:60]}")

## Step 4: Compare results

The key question: does the plan scan catch anything the static scan misses? Plan scanning sees resolved values (e.g., `var.environment` → `"dev"`), so it can evaluate resource names and interpolated values that static scanning treats as opaque.

In [ ]:
def compare_scans(static: list, plan: list) -> dict:
    """Compare failed checks between static and plan scans.
    
    Returns dict with overlapping and unique findings.
    """
    static_ids = {fc.get("check_id", "") for fc in static}
    plan_ids = {fc.get("check_id", "") for fc in plan}
    
    return {
        "static_only": static_ids - plan_ids,
        "plan_only": plan_ids - static_ids,
        "both": static_ids & plan_ids,
        "static_count": len(static),
        "plan_count": len(plan),
    }


comparison = compare_scans(static_failed, plan_failed)

print("=== Comparison ===")
print(f"Static scan failed checks:  {comparison['static_count']}")
print(f"Plan scan failed checks:    {comparison['plan_count']}")
print(f"Findings in both scans:     {len(comparison['both'])}")
print(f"Findings only in static:    {comparison['static_only']}")
print(f"Findings only in plan:      {comparison['plan_only']}")

if comparison['plan_only']:
    print("\nPlan scan caught these that static missed:")
    for check_id in sorted(comparison['plan_only']):
        print(f"  - {check_id}")
        
if comparison['static_only']:
    print("\nStatic scan caught these that plan missed:")
    for check_id in sorted(comparison['static_only']):
        print(f"  - {check_id}")

## Summary

**Static directory scan** is simpler — no Terraform CLI dependency, no plan generation step. It catches all policy violations it can detect from the raw source.

**Plan JSON scan** catches everything the static scan does *plus* anything that depends on resolved variable values. The trade-off: it requires `terraform init` + `terraform plan` to succeed first, which adds complexity and a dependency on the provider being available.

**Recommendation:** Use static scans for quick feedback loops (pre-commit, IDE), and plan scans for CI/CD gates where the full Terraform execution context is available anyway.

One thing I'm not sure about yet: how plan scans handle modules that haven't been downloaded yet (the `--download-external-modules` flag might matter differently for each mode).

In [ ]:
# Clean up temp directory
import shutil
shutil.rmtree(tf_dir, ignore_errors=True)
print(f"Cleaned up {tf_dir}")